In [3]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS
# ==============================================================================

# Ferramentas de Nuvem (Google Cloud Platform - GCP)
from google.cloud import bigquery  # Cliente oficial de comunicação com o Google BigQuery.
from google.oauth2 import service_account  # Responsável pela autenticação segura via Service Account JSON.

# Manipulação e Análise de Dados
import pandas as pd  # Biblioteca principal para tratamento tabular e análises de dados.
import pandas_gbq

# ==============================================================================
# 2. CONFIGURAÇÕES DE VISUALIZAÇÃO (PANDAS DISPLAY CONFIG)
# ==============================================================================

# Por padrão, o Pandas esconde colunas/linhas se a tabela for muito grande.
# Estas configurações garantem que possamos inspecionar os dados no terminal/notebook.
pd.set_option("display.max_columns", None)  # Não limita o número máximo de colunas exibidas na tela.
pd.set_option("display.max_rows", 10)  # Limita a exibição visual a no máximo 10 linhas para não travar a IDE.

In [4]:
# ==============================================================================
# 3. CONFIGURAÇÃO DE AUTENTICAÇÃO E CONEXÃO COM O BIGQUERY
# ==============================================================================

# Caminho local do arquivo JSON contendo as credenciais privadas da conta de serviço.
# ATENÇÃO: Nunca versionar este arquivo no GitHub público.
meu_service_account_file = "C:/Users/1592385/Documents/GitHub/data_project-template/secrets/meu-projeto-eg-dados.json"

# ID do projeto pessoal criado no Google Cloud Platform.
meu_gcp_project_id = "meu-projeto-eg-dados"

# Conversão do arquivo JSON em um objeto de credenciais autenticadas.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# Inicialização do cliente de comunicação com o BigQuery.
# Este objeto será utilizado para leitura e escrita de tabelas na nuvem.
bigquery_client = bigquery.Client(credentials=gcp_credentials, project=meu_gcp_project_id)

In [ ]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# ------------------------------------------------------------------------------
# Fonte: Camada Bronze - BigQuery
# Objetivo:
#   Carregar dados financeiros e comerciais de filmes da Marvel e DC.
#
# Dataset:
#   bronze.boxoffice_dc_marvel
# ------------------------------------------------------------------------------

df_boxoffice_dc_marvel = bigquery_client.query("""
    SELECT *
    FROM bronze.boxoffice_dc_marvel
    """).to_dataframe(create_bqstorage_client=False)

In [9]:
# ------------------------------------------------------------------------------
# Fonte: Camada Bronze - BigQuery
# Objetivo:
#   Carregar dados técnicos e avaliações do IMDB.
#
# Dataset:
#   bronze.imdb_filmes
# ------------------------------------------------------------------------------

df_imdb_filmes = bigquery_client.query("SELECT * FROM bronze.imdb_filmes ").to_dataframe(create_bqstorage_client=False)

In [10]:
# ==============================================================================
# 5. CONSOLIDAÇÃO DOS DATASETS (MERGE / JOIN)
# ==============================================================================

# Realizando junção entre os datasets financeiros e técnicos.
#
# Regras de cruzamento:
#   - FILM  -> TITLE
#   - YEAR  -> YEAR
#
# Tipo de JOIN:
#   - OUTER JOIN
#
# Objetivo:
#   Preservar todos os registros presentes em ambas as tabelas,
#   mesmo que existam inconsistências entre os datasets.

df_complete_movie_info = (
    pd.merge(
        df_boxoffice_dc_marvel,
        df_imdb_filmes,
        left_on=["FILM", "YEAR"],
        right_on=["TITLE", "YEAR"],
        how="outer",
    )
    .sort_values(by="YEAR", ascending=False)
    .drop_duplicates(keep="last")
)

In [17]:
df_complete_dc_marvel = df_complete_movie_info[
    (df_complete_movie_info["FRANCHISE"] == "Marvel") |
     (df_complete_movie_info["FRANCHISE"] == "DC")
]

In [18]:
df_complete_dc_marvel

,int64_field_0,FILM,US_RELEASE_DATE,BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BOX_OFFICE_GROSS_WORLDWIDE,BUDGET_x,MCU,PHASE,DISTRIBUTOR,MPAA_RATING,LENGTH,MINUTES,FRANCHISE,CHARACTER_FAMILY,DOMESTIC,GROSS_TO_BUDGET,ROTTEN_TOMATOES_CRITIC_SCORE,MALE_FEMALE_LED,YEAR,INFLATION_ADJUSTED_WORLDWIDE_GROSS,INFLATION_ADJUSTED_BUDGET,25X_PROD,BREAK_EVEN,Unnamed: 0,ID,TITLE,LINK,DURATION,RATING_MPA,RATING_IMDB,VOTE,BUDGET_y,GROSS_WORLD_WIDE,GROSS_US_CANADA,GROSS_OPENING_WEEKEND,DIRECTOR,WRITER,STAR,GENRE,COUNTRY_ORIGIN,FILMING_LOCATION,PRODUCTION_COMPANY,LANGUAGE,WIN,NOMINATION,OSCAR
15509,112,Madame Web,2024-02-14,42619699,54000000,96619699,80000000,False,NaN,Columbia Pictures,PG-13,01:56:00,116.0,Marvel,Spider-Man Allies and Villains,0.44,1.21,13.0,Female,2024.0,96619699,80000000,200000000,Flop,6287,tt11057302,Madame Web,https://www.imdb.com/title/tt11057302,1h 56m,PG-13,4.0,96000.0,80000000.0,100498764.0,43817106.0,15335860.0,SJ Clarkson,"Matt Sazama, Burk Sharpless, Claire Parker","Dakota Johnson, Sydney Sweeney, Isabela Merced","Psychological Thriller, Superhero, Action, Adv...","United States, Mexico","Palenque National Park, Chiapas, Mexico","Columbia Pictures, Di Bonaventura Pictures, Ma...","English, Spanish",0,0,0
3687,109,Blue Beetle,2023-08-18,72488072,56800000,129288072,120000000,False,NaN,Warner Bros.,PG-13,02:07:00,127.0,DC,Blue Beetle,0.56,1.08,78.0,Male,2023.0,129288072,120000000,300000000,Flop,1699,tt9362930,Blue Beetle,https://www.imdb.com/title/tt9362930,2h 7m,PG-13,5.9,105000.0,104000000.0,130788072.0,72488072.0,25030225.0,Angel Manuel Soto,Gareth DunnetAlcocer,"Xolo Mariduea, Bruna Marquezine, Becky G","Superhero, Action, Adventure, SciFi, Thriller","United States, Mexico, Canada, Germany",Puerto Rico,"Warner Bros, Warner Bros Pictures Mexico, DC E...","English, Spanish, Portuguese",0,12,0
27804,110,The Marvels,2023-11-10,84500223,121373601,205873824,219800000,True,5.0,Walt Disney Studios Motion Pictures,PG-13,01:45:00,105.0,Marvel,Captain Marvel,0.41,0.94,62.0,Female,2023.0,205873824,219800000,549500000,Flop,28636,tt10676048,The Marvels,https://www.imdb.com/title/tt10676048,1h 45m,PG-13,5.5,142000.0,220000000.0,206136825.0,84500223.0,46110859.0,Nia DaCosta,"Nia DaCosta, Megan McDonnell, Elissa Karasik","Brie Larson, Teyonah Parris, Iman Vellani","Superhero, Action, Adventure, Fantasy",United States,"Calabria, Italy",Marvel Studios,"English, Urdu",0,11,0
2074,111,Aquaman and the Lost Kingdom,2023-12-20,124436589,309900000,434336589,205000000,False,NaN,Warner Bros.,PG-13,02:04:00,124.0,DC,Aquaman,0.29,2.12,35.0,Male,2023.0,434336589,205000000,512500000,Flop,17186,tt9663764,Aquaman and the Lost Kingdom,https://www.imdb.com/title/tt9663764,2h 4m,PG-13,5.6,102000.0,205000000.0,439381226.0,124481226.0,27686211.0,James Wan,"David Leslie JohnsonMcGoldrick, James Wan, Jas...","Jason Momoa, Patrick Wilson, Yahya AbdulMateen II","Action Epic, Superhero, Action, Adventure, Fan...","United States, United Kingdom, Canada, Austral...","Saunton Sands, Devon, England, UK","Warner Bros, DC Entertainment, Domain Entertai...",English,0,6,0
1981,104,Ant-Man and the Wasp: Quantumania,2023-02-17,214504909,261566271,476071180,200000000,True,5.0,Walt Disney Studios Motion Pictures,PG-13,02:04:00,124.0,Marvel,Ant-Man,0.45,2.38,46.0,Co-starring,2023.0,476071180,200000000,500000000,Flop,7202,tt10954600,Ant-Man and the Wasp: Quantumania,https://www.imdb.com/title/tt10954600,2h 4m,PG-13,6.0,244000.0,200000000.0,476071180.0,214504909.0,106109650.0,Peyton Reed,"Jeff Loveness, Stan Lee, Larry Lieber","Paul Rudd, Evangeline Lilly, Michael Douglas","Superhero, Action, Adventure, Comedy, SciFi","United States, Australia, Canada","Nevsehir, Turkey","Marvel Studios, Lightnin Production Rentals, Q...",English,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11688,4,Howard th

In [19]:
# ==============================================================================
# 7. REMOÇÃO DE COLUNAS REDUNDANTES E DESNECESSÁRIAS
# ==============================================================================

# Eliminando colunas:
#   - duplicadas após o merge,
#   - auxiliares,
#   - pouco relevantes para análises finais,
#   - ou já representadas por outras métricas equivalentes.

df_complete_dc_marvel = df_complete_dc_marvel.drop(
    columns=[
        "25X_PROD",
        "BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA",
        "BOX_OFFICE_GROSS_WORLDWIDE",
        "BUDGET_y",
        "DURATION",
        "LENGTH",
        "MPAA_RATING",
        "TITLE",
        "YEAR",
    ]
).reset_index(drop=True)

In [21]:
# ==============================================================================
# 8. VALIDAÇÃO EXPLORATÓRIA DOS DADOS
# ==============================================================================

# Verificando os filmes com maior nota no IMDB após o processo de consolidação.
# Esta análise rápida ajuda a validar:
#   - integridade do merge,
#   - consistência das notas,
#   - ordenação correta dos dados.

df_complete_dc_marvel.sort_values(by="RATING_IMDB", ascending=False).head(2)

,int64_field_0,FILM,US_RELEASE_DATE,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BUDGET_x,MCU,PHASE,DISTRIBUTOR,MINUTES,FRANCHISE,CHARACTER_FAMILY,DOMESTIC,GROSS_TO_BUDGET,ROTTEN_TOMATOES_CRITIC_SCORE,MALE_FEMALE_LED,INFLATION_ADJUSTED_WORLDWIDE_GROSS,INFLATION_ADJUSTED_BUDGET,BREAK_EVEN,Unnamed: 0,ID,LINK,RATING_MPA,RATING_IMDB,VOTE,GROSS_WORLD_WIDE,GROSS_US_CANADA,GROSS_OPENING_WEEKEND,DIRECTOR,WRITER,STAR,GENRE,COUNTRY_ORIGIN,FILMING_LOCATION,PRODUCTION_COMPANY,LANGUAGE,WIN,NOMINATION,OSCAR
75,36,The Dark Knight,2008-07-18,472349861,185000000,False,NaN,Warner Bros.,152.0,DC,Batman,0.53,5.45,94.0,Male,1425605587,261816105,Success,3230,tt0468569,https://www.imdb.com/title/tt0468569,PG-13,9.0,3000000.0,1.009054e+09,534987076.0,158411483.0,Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S Goyer","Christian Bale, Heath Ledger, Aaron Eckhart","Action Epic, Epic, Superhero, Action, Crime, D...","United States, United Kingdom","Chicago, Illinois, USA","Warner Bros, Legendary Entertainment, Syncopy","English, Mandarin",0,164,0
6,107,Spider-Man: Across the Spider-Verse,2023-06-02,309205354,100000000,False,NaN,Columbia Pictures,140.0,Marvel,Spider-Man,0.55,6.91,96.0,Male,690516673,100000000,Success,25318,tt9362722,https://www.imdb.com/title/tt9362722,PG,8.5,428000.0,6.908247e+08,381593754.0,120663589.0,"Joaquim Dos Santos, Kemp Powers, Justin K Thom...","Phil Lord, Christopher Miller, Dave Callaham","Shameik Moore, Hailee Steinfeld, Brian Tyree H...","Computer Animation, Superhero, Supernatural Fa...",United States,NaN,"Columbia Pictures, Marvel Entertainment, Avi A...",English,0,164,1


In [22]:
# ==============================================================================
# 9. CÓPIA DE SEGURANÇA PARA TRATAMENTO (DATA CLEANING)
# ==============================================================================

# Criando uma cópia independente do DataFrame consolidado.
# Isso evita modificar acidentalmente o dataset original durante os tratamentos.

df_tratado = df_complete_dc_marvel.copy()

In [23]:
# ==============================================================================
# 10. TRATAMENTO DE COLUNAS MONETÁRIAS
# ==============================================================================

# Colunas contendo valores financeiros armazenados originalmente como texto.
#
# Problemas encontrados:
#   - símbolo "$"
#   - separadores de milhar ","
#   - valores nulos representados como string "None"
#
# Objetivo:
#   Converter tudo para tipo numérico float.

money_cols = [
    "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
    "BUDGET_x",
    "INFLATION_ADJUSTED_BUDGET",
    "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
    "GROSS_OPENING_WEEKEND",
    "GROSS_US_CANADA",
    "GROSS_WORLD_WIDE",
]

for col in money_cols:
    df_tratado[col] = (
        df_tratado[col]
        .astype(str)
        .str.replace("$", "")
        .str.replace(",", "")
        .replace("None", 0)
        .astype(float)
    )

In [25]:
# ==============================================================================
# 11. TRATAMENTO DE COLUNAS PERCENTUAIS
# ==============================================================================

# Removendo o símbolo "%" e convertendo os valores para float.

percent_cols = ["DOMESTIC"]

for col in percent_cols:
    df_tratado[col] = df_tratado[col].astype(str).str.replace("%", "").astype(float)

In [26]:
# ==============================================================================
# 12. PADRONIZAÇÃO DE COLUNAS DE DATA
# ==============================================================================

# Conversão de strings para formato datetime.
#
# Formato original:
#   DD/MM/YYYY

date_cols = ["US_RELEASE_DATE"]

for col in date_cols:
    df_tratado[col] = pd.to_datetime(df_tratado[col], format="%d/%m/%Y")

In [27]:
# ==============================================================================
# 13. TRATAMENTO DE MÉTRICAS INTEIRAS
# ==============================================================================

# Colunas quantitativas contendo:
#   - votos,
#   - premiações,
#   - indicações,
#   - métricas críticas.
#
# Estratégia:
#   - preencher nulos com zero,
#   - converter para inteiro.

int_cols = [
    "VOTE",
    "WIN",
    "OSCAR",
    "NOMINATION",
    "ROTTEN_TOMATOES_CRITIC_SCORE",
]

for col in int_cols:
    df_tratado[col] = df_tratado[col].fillna(0).astype(int)

In [28]:
df_tratado["YEAR"] = df_tratado["US_RELEASE_DATE"].dt.year

In [29]:
# ==============================================================================
# 14. NORMALIZAÇÃO DE CAMPOS TEXTUAIS
# ==============================================================================

# Alguns campos possuem múltiplos valores separados por vírgula.
#
# Exemplo:
#   "Action, Adventure, Sci-Fi"
#
# Estratégia:
#   - manter apenas o primeiro valor principal,
#   - remover espaços excedentes.

df_tratado["GENRE"] = df_tratado["GENRE"].str.split(",").str[0].str.strip()

df_tratado["COUNTRY_ORIGIN"] = df_tratado["COUNTRY_ORIGIN"].str.split(",").str[0].str.strip()

df_tratado["LANGUAGE"] = df_tratado["LANGUAGE"].str.split(",").str[0].str.strip()

df_tratado["PRODUCTION_COMPANY"] = df_tratado["PRODUCTION_COMPANY"].str.split(",").str[0].str.strip()

df_tratado["WRITER"] = df_tratado["WRITER"].str.split(",").str[0].str.strip()

In [30]:
# ==============================================================================
# 15. HIGIENIZAÇÃO FINAL DO DATAFRAME
# ==============================================================================

# Removendo colunas auxiliares desnecessárias.
#
# errors="ignore":
#   evita erro caso a coluna não exista no dataset.

df_tratado = df_tratado.drop(columns=["Unnamed: 0"], errors="ignore")

# Reorganizando colunas em ordem alfabética
# para padronização estrutural.

df_tratado = df_tratado[sorted(df_tratado.columns)].drop_duplicates()

In [34]:
df_tratado.head(3)

,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BREAK_EVEN,BUDGET_x,CHARACTER_FAMILY,COUNTRY_ORIGIN,DIRECTOR,DISTRIBUTOR,DOMESTIC,FILM,FILMING_LOCATION,FRANCHISE,GENRE,GROSS_OPENING_WEEKEND,GROSS_TO_BUDGET,GROSS_US_CANADA,GROSS_WORLD_WIDE,ID,INFLATION_ADJUSTED_BUDGET,INFLATION_ADJUSTED_WORLDWIDE_GROSS,LANGUAGE,LINK,MALE_FEMALE_LED,MCU,MINUTES,NOMINATION,OSCAR,PHASE,PRODUCTION_COMPANY,RATING_IMDB,RATING_MPA,ROTTEN_TOMATOES_CRITIC_SCORE,STAR,US_RELEASE_DATE,VOTE,WIN,WRITER,YEAR,int64_field_0
0,54000000.0,Flop,80000000.0,Spider-Man Allies and Villains,United States,SJ Clarkson,Columbia Pictures,0.44,Madame Web,"Palenque National Park, Chiapas, Mexico",Marvel,Psychological Thriller,15335860.0,1.21,43817106.0,100498764.0,tt11057302,80000000.0,96619699.0,English,https://www.imdb.com/title/tt11057302,Female,False,116.0,0,0,NaN,Columbia Pictures,4.0,PG-13,13,"Dakota Johnson, Sydney Sweeney, Isabela Merced",2024-02-14,96000,0,Matt Sazama,2024,112
1,56800000.0,Flop,120000000.0,Blue Beetle,United States,Angel Manuel Soto,Warner Bros.,0.56,Blue Beetle,Puerto Rico,DC,Superhero,25030225.0,1.08,72488072.0,130788072.0,tt9362930,120000000.0,129288072.0,English,https://www.imdb.com/title/tt9362930,Male,False,127.0,12,0,NaN,Warner Bros,5.9,PG-13,78,"Xolo Mariduea, Bruna Marquezine, Becky G",2023-08-18,105000,0,Gareth DunnetAlcocer,2023,109
2,121373601.0,Flop,219800000.0,Captain Marvel,United States,Nia DaCosta,Walt Disney Studios Motion Pictures,0.41,The Marvels,"Calabria, Italy",Marvel,Superhero,46110859.0,0.94,84500223.0,206136825.0,tt10676048,219800000.0,205873824.0,English,https://www.imdb.com/title/tt10676048,Female,True,105.0,11,0,5.0,Marvel Studios,5.5,PG-13,62,"Brie Larson, Teyonah Parris, Iman Vellani",2023-11-10,142000,0,Nia DaCosta,2023,110


In [35]:
# ==============================================================================
# 16. CARGA ANALÍTICA - CAMADA SILVER (LOAD)
# ==============================================================================

# Salvando o dataset tratado na camada Silver do Data Warehouse.
#
# Camada Silver:
#   - dados limpos,
#   - estruturados,
#   - prontos para análises,
#   - dashboards,
#   - machine learning,
#   - ou consumo analítico.

pandas_gbq.to_gbq(
    df_tratado,
    project_id=meu_gcp_project_id,
    destination_table="silver.complete_dc_marvel",
    if_exists="replace",
    credentials=gcp_credentials,
)